# RESDY Tutorial

**Authors**: George Weston, Matteo Degiacomi

This Jupyter notebook features a guide which can be used for using the RESDY package to go from an organism code of uniprot code through to a set of curated structures and a measurements dataframe providing an extensive featurisation of properties of side chains of amino acids. The output given from this provides a perfect base for downstream classification tasks.

* 1. [Data Gathering](#gather)
    - 1.1. [Mine UNIPROT database](#uniprot)
    - 1.2. [Mine PDB and AlphaFold databases](#pdb)
* 2. [Feature Extraction](#measure)
* 3. [Data Analysis](#analysis)
    - 3.1. [Loading and Cleaning the Data](#analysis_load)
    - 3.2. [Data Aggregation and Plotting](#analysis_aggregate)
    - 3.3. [GO Terms Extraction](#analysis_go)
* 4. [Interactive Data Visualization](#plot)

Further explanations about each of these steps are provided in the individual sections below. The demonstration of the codebase in these steps does not cover every possible option that can be used within the software, though should be able to give you a rough idea of what is possible. For full details of the software, please see the API.

Let's get started by executing the cell below (select it and press the <kbd>&#x21E7; Shift</kbd> + <kbd>&#x23ce; Enter</kbd>keys) to import all the Python packages required. If you have already installed the package, please see the desciption in the README.md file on the homepage of the github repo.

In [ ]:
import os
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
from ipywidgets import widgets

import resdy as RD

Before getting started, we should also define an output directory. This will be the location for all output files created throughout this code and will be created in the working directory.

If it is not defined, or none is passed, the default name will be "result".

In [ ]:
outdir = "demo"

## 1. Data Gathering  <a class="anchor" id="gather"></a>

### 1.1. Mine UNIPROT database <a class="anchor" id="uniprot"></a>

We will start by compiling a database of protein data of interest by mining the [UNIPROT database](https://www.uniprot.org/). This can be done by either providing an input csv file, or by directly providing UNIPROT codes. Methods to automatically fetch UNIPROT codes for a whole organism are available and demostrated below too.

The main commands enabling information gathering to compile the database are `Uniprot.from_csv_file` and `Uniprot.get_protein_data`. These can be launched multiple times, to compile a database saved in a pandas DataFrame (`Uniprot.df`).

Generally, you will only use one of the following methods at once, however multiple can be used and the results are appended onto each other, duplicates are automatically dropped to prevent duplication downstream.

#### 1.1.1. Selection using csv files 

We will start by loading a list of UNIPROT codes contained in a short text file. Let's have a look at the file itself before loading it into our pipeline.

Note: If you are using your own csv file, the headers should follow the format of 'Uniprot_Entry' for the uniprot codes, 'PDB_Code' if you wish to specify specific PDB codes along with this and 'Resid' if you wish to record a list of modified residues for later use.

In [ ]:
df = pd.read_csv(f'{outdir}{os.sep}demo_input.csv')
df['Uniprot_Entry'] #show data





As you can see, the file simply contains a list of UNIPROT codes. Let's look for the data associated with them by mining UNIPROT!

In [ ]:
UP = RD.Uniprot()
UP.from_csv_file(f'{outdir}{os.sep}demo_input.csv')
UP.df






#### 1.1.2. Selection via manual insertion 

Besides loading UNIPROT codes from a text file, we can also append hand picked proteins in the database, by providing their UNIPROT code. Every time the command `Uniprot.get_protein_data`, data is appended to the DataFrame of previously gathered proteins.

In [ ]:
UP = RD.Uniprot()
UP.get_protein_data('Q03513')
UP.df

#### 1.1.3. Whole organism selection

If you are interested in fetching all the UNIPROT codes associated with a specific organism, the methods `Uniprot.get_organism_proteins` is available. It takes as parameter a proteome ID, and returns a list of UNIPROT codes. The list can be then iterated on, to load information on each identified protein. Optionally, the parameter `reviewed_only=True` can be used to ensure only codes of reviewed proteins are retrieved. Before calling this method though, it may be a good idea to determine how many proteins codes will be returned, usin the method `Uniprot.get_organism_proteins`.

*What follows is an example. In the rest of this tutorial, we will work on the data previously fetched*.

We will look for all the proteins in the Aravan virus (ARAV) proteome, code: [UP000007445](https://www.uniprot.org/proteomes/UP000007445).  Let's start by determining how many proteins are associated with this organism, and how many of these are reviewed.

In [ ]:
UP_organism = RD.Uniprot()
count_all_prot = UP_organism.count_organism_proteins('UP000007445', reviewed_only=False)
count_reviewed_prot = UP_organism.count_organism_proteins('UP000007445', reviewed_only=True)
print(f'total: {count_all_prot}, reviewed: {count_reviewed_prot}')

Unsurprisingly, only a fraction of all proteins are reviewed. Let's only gather these ones. The `Uniprot.get_organism_proteins` method will return a list of relevant UNIPROT codes, and print progress text while it runs (i.e., "number of fetched UNIPROT codes / total number of codes to fetch").

In [ ]:
data = UP_organism.get_organism_proteins('UP000007445', reviewed_only=True)
data

If you want to see how infromation is retrieved from a list of UNIPROT codes, uncomment the lines below.
If not, there is no need to run this cell.

<div class="alert alert-warning">
<b>Warning:</b> For a whole organism, incuding unreviewed proteins, this can take a long time!
</div>

In [ ]:
UP.from_organism('UP000007445', reviewed_only=True)  

### 1.2. Mine PDB and AlphaFold databases  <a class="anchor" id="pdb"></a>

After gathering a list of Uniprot codes (contained in the Uniprot DataFrame - `Uniprot.df`), conformations for all proteins described need to be downloaded and curated.
Files will be downloaded from the [Protein Data Bank (PDB)](https://www.rcsb.org/) and the [AlphaFold database](https://alphafold.ebi.ac.uk/).
There is 1 key optional parameters which should be passed to the PDB constructor, `outdir` (output directory, "result" by default). There are many other arguments which can be passed all for differnet reasons such as the largest acceptable gap in sequence (`gap`, 10 by default) or whether to perform a minimisation on structures (`minimise_strucs`, set to 'AF' by default to only minimise AF structures). There are many more, full details of options can be found in the API.

Proteins having a consecutive number of missing aminoacids greater than this cutoff value will be disregarded. Note this codebase, due to restrictions placed on the legacy PDB data format will not curate any protein structures which either contain double letter chain names or more than 9999 residues in a chain.

For each entry in the input DataFrame:
* raw files from the PDB and Alphafold databases are downloaded in a subfolder within the output directory (defined by `PDB.raw_dir`, "conformations" by default)
* each raw file is cleaned: heteroatoms, hydrogens, and non-protein non-standard aminoacids are amended. Ions are preserved, as they may have important effect on the later feature extraction phase
* different conformations (usually from NMR ensembles) are saved in different files
* if multiple aminoacid rotamers exist, multiple files are produced (one per rotamer)
* all produced files are patched, i.e. missing parts in the sequence are added using Modeller
* all files correctly patched are saved in a different subfolder in the output directory (defined by `PDB.curated_dir`, "curated" by default).
* if insertion codes are present, these are noted and the structure is not carried through
* changes key modified back to regular state (note all modified residues can be reverted if `remove_all_modifications` is set to True)
* adds in element codes if these are missing

All of these steps are completed to ensure that all structures are whole, correct and able to have proper features calculated on for future analysis.
This process will generate a new DataFrame, `PDB.df`, containing information on all the structures that have been correctly curated.

<div class="alert alert-warning">
<b>Warning:</b> depending on how many proteins have to be downloaded and curated, this operation may take a long time. AlphaFold and PDB files not needing curation are processed in few seconds. Large multimers requiring curation can take up to 1 min. Multiple conformations in a single protein structure are individually curated.
</div>

In [ ]:
pdb = RD.PDB(outdir=outdir, gap=10, parallel=True)
pdb.gather_proteins(UP.df)

Show the DataFrame reporting on structures successfully downloaded and curated, and save it into a CSV file. 

A filename can be given via the `outname` option (eg `pdb.save_state(outname = proteins.csv)`). If no filename is given, file `proteins.csv` will be saved in the output directory.

In [ ]:
pdb.save_state()
pdb.df

If you have already gathered protein data and wish to load the CSV file with the protein information to continue, this can be done with the method `PDB.load_state(filename)`.

In [ ]:
pdb = RD.PDB(outdir=outdir, gap=10)
pdb.load_state(fname=f'{outdir}{os.sep}proteins.csv')
pdb.df

## 2. Feature Extraction  <a class="anchor" id="measure"></a>

After structures have been curated through the previous steps within this workbook, features can be calculated for the lysines within the proteins.
The `Measure` class is used for this, when using this a log file will be created within `outdir`. The filename is defined by the parameter `log_path`, and is "measure_log.txt" by default.
The following features are currently available for calculation:

<style>
.table_component {
    overflow: auto;
    width: 100%;
}

.table_component table {
    border: 1px solid #dededf;
    height: 100%;
    width: 100%;
    table-layout: auto;
    border-collapse: collapse;
    border-spacing: 1px;
    text-align: left;
}

.table_component caption {
    caption-side: top;
    text-align: left;
}

.table_component th {
    border: 1px solid #dededf;
    background-color: #eceff1;
    color: #000000;
    padding: 5px;
}

.table_component td {
    border: 1px solid #dededf;
    background-color: #ffffff;
    color: #000000;
    padding: 5px;
}
</style>
<div class="table_component" role="region" tabindex="0">
<table>
    <caption><br></caption>
    <thead>
        <tr>
            <th>Feature Name<br></th>
            <th>Details</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>PROPKA<br></td>
            <td>Calculates the pKa of the residue of interest using the PROPKA3 software.</td>
        </tr>
        <tr>
            <td>PKAANI*<br></td>
            <td>Calculates the pKa of the residue of interest using the pkaANI software.</td>
        </tr>
        <tr>
            <td>DEPTH</td>
            <td>Calculates the minimum depth of the terminus of the side chain of the residue of interest from the surface of the protein using the depth method from biobox.</td>
        </tr>
        <tr>
            <td>SASA<br></td>
            <td>Calculates the sovlvent accessible surface area (SASA) of the terminus of the side chain of the residue of interest within the protein using the Shrake-Rupley (rolling ball) method from biobox.</td>
        </tr>
        <tr>
            <td>AEVs<br></td>
            <td>Uses the ANI-2x software to produce atomic environment vectors (AEVs) to represent both local environment of the side chain of the residue of interest. This encodes the structure and the neighbouring atoms into a 1008 value vector.</td>
        </tr>
        <tr>
            <td>DAS<br></td>
            <td>Calculates the dynamically accessible surface (DAS) area of the terminus of the side chain of the residue of interest within the protein using the __get_half_spheres() method from the Xlink class in biobox.</td>
        </tr>
        <tr>
            <td>LEGOLAS*<br></td>
            <td>Calculates the 15N nmr shift of the backbone N in the residue of interest.</td>
        </tr>
        <tr>
            <td>MELODIA<br></td>
            <td>Calculates the structural information for the residue of interest's conformation using melodia-py.</td>
        </tr>
        <tr>
            <td>SECONDARYSTRUCTURE<br></td>
            <td>Calculates the secondary structure of the residue of interest using biopython and dssp.</td>
        </tr>
        <tr>
            <td>FRUSTRATION,DENSITY<br></td>
            <td>Calculates the frustration and density values of the residue of interest using frustratometer.</td>
        </tr>
        <tr>
            <td>SEQCHARGE<br></td>
            <td>Calculates the sequence charge of a sequence window around the residue of interest.</td>
        </tr>
        <tr>
            <td>RMSF<br></td>
            <td>Calculates the root mean square fluctuation of a residue of interest across all possible conformations curated using biobox.</td>
        </tr>
        <tr>
            <td>EVOLUTION<br></td>
            <td>Uses the ESM model to calculate a vector representing the evolution of the sequence.</td>
        </tr>
        <tr>
            <td>FLEXIBILITY<br></td>
            <td>Extracts the B-factors from the PDB files, normalised to take account for structure resolution.</td>
        </tr>
    </tbody>
</table>
</div>

Note these are all possible features that are shipped as possible to run. Features with a * associated with them are coded in to be run, but the packages required for these are not listed in the environment given and will need extra setup from the user before running. The whole software is modular and designed that if you would like to add a new feature, it is very easy to do so. Simply take the FEATURE.py template, create your feature and call it using the same name.

To define which features to calculate, use the features option in setting up the class. Format required is a dataframe where the values for each feature key is another dataframe giving optional inputs for the specific feature. All features are setup such that they will run as is, however customisation is possible. Example input: `features={'propka': {}, 'sasa': {}, 'depth': {'calculation_type': 'ResidDepth'}, 'aev': {}, 'das': {}}`. If you wish for all of the features to be calculated, 'all' can be passed which will calculate all possible features with the setup available.

The final results are saved into a CSV file. The name can be defined using the `outname` option, if no filename is given, `measures.csv` is used. This is saved into a folder defined by `outdir` ("result" by default). 

<div class="alert alert-warning">
<b>Warning:</b>  measurement calculations typically take anywhere between 1 second and 5 minutes per curated protein structure. The time depends on the structure size and the number of lysines within the protein.
</div>

In [ ]:
M = RD.Measure(pdb.df, outdir=outdir, activate_log=True, log_path="measure_log.txt")
M.measure_data()
M.save_state()

<div class="alert alert-info">
<b>Note</b>: the <code>Measure</code> object features methods for calculations and data recovery:
    
* <code>M.restart_measure()</code> restarts the operations of <code>M.measure_dataframe()</code>, if these were interrupted by any reason.
* <code>M.recover_from_log(filename)</code> parses a logfile and returns a DataFrame.</div>


The data acquired during calculations is located in the `M.df` DataFrame. We can sort the data according to any key of interest, for instance UNIPROT entry followed by resid.

In [ ]:
M.df.sort_values(['Uniprot_Entry', 'Resid'])

## 3. Data Analysis  <a class="anchor" id="analysis"></a>

### 3.1. Loading and cleaning the data   <a class="anchor" id="analysis_load"></a>

The Analysis class is responsible for interpreting the lysine features we extracted. Creating `Analysis` requires a DataFrame containing data gathered by a `Measure` instance. This can come either from a previously saved logfile (as we shall do in the next box), or from a DataFrame in a `Measure` instance (`Measure.df`). Below, we will load data from the CSV file file we previously saved.

In [ ]:
df = pd.read_csv(f'Demo{os.sep}measures.csv')
analysis = RD.Analysis(df, outdir=outdir)

The loaded measurement data is now internally stored in the `Analysis.df` DataFrame.

Some lysine measurements may be suspiciously high (e.g., pKa>14) or low (e.g., pKa<1). This could be an indication that, in their specific circumstance, calculations were faulty. These can be identified as follows, where the output is a new DataFrame.

In [ ]:
df_suspicious = analysis.get_extreme_values(feature='propka', lower=1, upper=14)

It is also possible to remove any rows from the dataframe that are not specific residues of interest. Simply pass a dataframe containing columns of UNIPROT codes and a resid column with associated resid number for that UNIPROT. If there are multiple of interest per UNIPROT, write multiple entries to the CSV until are are covered. This will save a new dataframe of the name `measures_cut.csv`.

In [ ]:
required_residues = f'{outdir}{os.sep}demo_input.csv'
df_cut = analysis.remove_not_important_residues(req_resid_table=required_residues)
df_cut

### 3.2. Data Aggregation and Analysis   <a class="anchor" id="analysis_aggregate"></a>

#### 3.2.1 Aggregation

The data we have now cleaned can be aggregated. Aggregation allows you to create statistical data representing the conformational landscape for every site of residue of interest for each feature (a single residue of interest may feature multiple measurements created, it is important for downstream processing after this programme has been used that there is only 1 set of data per residue of interest). This is particularly important to prevent any bias or potential data leakage in downstream ML classification tasks. This operation is carried out by calling `Aggregation.aggregate()`, and the data is internally stored in the DataFrame `Aggregation.df_agg`.

The aggregation class has several methods for completing this to allow full analysis of the conformational landscape of the residues of interest. Parameter options:
* aggregation method -> defines which statistic values should be returned for each feature, default `minmaxavg`
* feature_to_include -> list of features to get statistics on, defaults to all that are present in the measures dataframe
* aev_red_method -> method to reduce the dimensions of the AEVs to reduce data clouding.

There are many options for all of these variables, see API docs on aggregation.py for full details.


In [ ]:
agg = RD.Aggregation(df, aggregation_method='minmaxavg',
                     outdir=outdir,
                      features_to_include=['propka', 'sasa', 'depth'],
                      aev_red_method='pca')
agg.aggregate_data()
agg.df_agg

One alternative method to aggregate data is, for each residue of interest to find a row of data which contains the best mix of feature data comparing the different requirements of each feature. Eg conformation which has the lowest depth, lowest pKa and highest SASA. Weights can also be passed to the function to bias the selection of a row of data for the residue of interest towards the metrics in different ways. The default weights given are even for each.The result of this aggregation will be stored in the DataFrame `Analysis.df_sub`.


In [ ]:
df = pd.read_csv(f'Demo{os.sep}measures.csv')
analysis = RD.Analysis(df, outdir=outdir)
analysis.relative_best(features=['propka', 'depth', 'sasa'], weights=[0.5, 0.8, 0.3])
analysis.df_sub

#### 3.2.2 AlphaFold vs PDB data

An interesting question involves comparing AlphaFold data to PDB data. The following method enables selecting only the data from structures predicted by AlphaFold. Note that any DataFrame in `Analysis` can always be interrogated with pandas syntax, e.g., via the [query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) method.

In [ ]:
df_af = analysis.get_data_alphafold()
df_af

#### 3.2.3 Basic Plotting of Data

`Analysis` class features two methods which are able to produce plots for quick analysis of the output data. These include histograms of all the aggregated data in which you can choose the aggregation method to use and violin plots for a particular set of features for as many aggregation methods as you would like. For further information on the customisation of these graphs, please see the API. Here are some examples of this:

In [ ]:
analysis.plot_feature_histogram(plot_type='single', feature='depth', save_name='hist_depth.svg')
analysis.plot_feature_histogram(plot_type='all', save_name='hist_all.svg')
analysis.plot_feature_histogram(plot_type='agg', feature='depth', save_name='hist_agg.svg')
analysis.plot_feature_violins(features='depth', save_name='violin_depth_allagg.svg')
analysis.plot_feature_violins(agg_type='all', save_name='violin_all_features_all.svg')

### 3.3. GO Terms Extraction  <a class="anchor" id="analysis_go"></a>

Every protein is associated to a set of [Gene Ontology (GO) Terms](http://geneontology.org/). `Analysis` enables scraping these terms from the UNIPROT database. The data will be stored in a dictionary inside the `Analysis` class. *Technical note: this operation uses multi-threading for speed up.*

In [ ]:
analysis.GO_get_data()

Once the GO Terms of all the analysed proteins have been scraped, they can be interrogated with two different methods.

The first method searches for all the protein structures matching a specific GO code. Below, we will look for all lysines belonging to proteins associated with the GO term [0042803](https://www.ebi.ac.uk/QuickGO/term/GO:0042803) ("*dimerization activity*").

In [ ]:
analysis.GO_search_term(analysis.df, code ='0042803', name ='')

The second method extracts the GO Terms of a specific protein, defined by its UNIPROT code. Below, we retrieve all the GO termns of the protein P73832.

In [ ]:
GO_terms = analysis.GO_search_protein('P06733')
GO_terms

We will now look for enrichment of GO terms within a specific pKa and SASA region. This enables us to determine whether specific GO terms are more prevalent in specific regions of the pKa/SASA space.

In [ ]:
pka_range = [1.0, 9.3]
sasa_range = [21.9, 100.0]
df = analysis.enrichment_analysis(pka_range, sasa_range)
df

## 4. Interactive Data Visualization  <a class="anchor" id="plot"></a>

To view all the data created in this process, an interactive data visualisation procedure is provided here with plotly dash. The window can be launched via the code below and allows you to explore the vast array of data that has been created and start to directly compare the data.

In [ ]:
V = RD.Viewer(analysis, outdir=outdir)
V.launch_viewer()

---